# PSASE Lab

# Experiment 2: Multiple Linear Regression

This is our Experiment 2, we will implement **Multiple Linear Regression** to predict house prices using multiple features (size, bedrooms, floors, age).


# Outline
- [ 1 - Packages ](#1)
- [ 2 - Problem Statement ](#2)
- [ 3 - Dataset ](#3)
  - [ 3.1 View the variables ](#3.1)
  - [ 3.2 Visualize the data ](#3.2)
- [ 4 - Multiple Linear Regression Model ](#4)
  - [ 4.1 Notation ](#4.1)
  - [ 4.2 Single Prediction (vectorized using np.dot) ](#4.2)
- [ 5 - Compute Cost ](#5)
- [ 6 - Gradient Descent ](#6)
  - [ 6.1 Compute Gradient ](#6.1)
  - [ 6.2 Gradient Descent Implementation ](#6.2)
- [ 7 - Gradient Descent on Raw Features ](#7)
- [ 8 - Feature Scaling ](#8)
  - [ 8.1 Z-Score Normalization ](#8.1)
  - [ 8.2 Normalize the Training Data ](#8.2)
  - [ 8.3 Gradient Descent with Normalized Features ](#8.3)
  - [ 8.4 Convergence Comparison ](#8.4)
- [ 9 - Predictions ](#9)

<a name="1"></a>
## 1 - Packages

First, let's run the cell below to import all the packages that we will need during this experiment.
- [numpy](www.numpy.org) is the fundamental package for working with matrices in Python.
- [matplotlib](http://matplotlib.org) is a famous library to plot graphs in Python.
- [plotly](https://plotly.com/python/) is used for interactive 3D visualizations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import math
%matplotlib inline

np.set_printoptions(precision=2)

<a name="2"></a>
## 2 - Problem Statement

Suppose you are a real estate agent and you want to predict the price of houses based on their features.

You have collected data on houses that were recently sold, and you have the following features for each house:

| Feature | Description | Units |
|:--------|:------------|:------|
| Size | Area of the house | sqft |
| Bedrooms | Number of bedrooms | count |
| Floors | Number of floors | count |
| Age | Age of the house | years |
| **Price** | **Sale price (target)** | **1000s of dollars** |

Can you use the data to help you predict the price for new houses based on their features?

Unlike simple linear regression where we had just one feature (e.g., population → profit), here we have **multiple features** ($n = 4$), so we need **multiple linear regression**.

## 3 - Dataset

Let's load the dataset from `data/houses.txt`. The file contains 200 examples with 4 features and 1 target variable.

In [ ]:
# Load the dataset
data = np.loadtxt('data/houses.txt', delimiter=',')
X_train = data[:, :4]   # Features: size, bedrooms, floors, age
y_train = data[:, 4]    # Target: price (in 1000s of dollars)

print(f"Dataset loaded successfully with {X_train.shape[0]} examples.")

<a name="3.1"></a>
### 3.1 View the variables

Before starting on any task, it is useful to get more familiar with the dataset.  
A good place to start is to just print out each variable and see what it contains.

Note that unlike simple linear regression, our feature matrix $\mathbf{X}$ is now a **2D array** with shape $(m, n)$ where:
- $m$ = number of training examples
- $n$ = number of features

In [ ]:
# Print X_train
print("Type of X_train:", type(X_train))
print("Shape of X_train:", X_train.shape)
print("First five rows of X_train:\n", X_train[:5])

Each row of `X_train` is a vector $\mathbf{x}^{(i)}$ containing the 4 features for the $i^{th}$ house:
- Column 0: Size (sqft)
- Column 1: Bedrooms
- Column 2: Floors
- Column 3: Age (years)

Now, let's print `y_train`:

In [ ]:
# Print y_train
print("Type of y_train:", type(y_train))
print("Shape of y_train:", y_train.shape)
print("First five elements of y_train:\n", y_train[:5])

In [ ]:
# Dimensions
print('The shape of X_train is:', X_train.shape)
print('The shape of y_train is:', y_train.shape)
print('Number of training examples (m):', X_train.shape[0])
print('Number of features (n):', X_train.shape[1])

<a name="3.2"></a>
### 3.2 Visualize the data

With multiple features, we can create scatter plots for each feature vs. the target (price) to understand the relationships.

In [ ]:
feature_names = ['Size (sqft)', 'Bedrooms', 'Floors', 'Age (years)']

fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
for i in range(4):
    axes[i].scatter(X_train[:, i], y_train, marker='x', c='r')
    axes[i].set_xlabel(feature_names[i])
    if i == 0:
        axes[i].set_ylabel('Price (in $1000s)')
    axes[i].set_title(f'Price vs {feature_names[i]}')
plt.tight_layout()
plt.show()

Our goal is to build a **multiple linear regression** model to fit this data.
- With this model, you can then input a new house's features (size, bedrooms, floors, age), and have the model estimate its price.

<a name="4"></a>
## 4 - Multiple Linear Regression Model

<a name="4.1"></a>
### 4.1 Notation

Here is a summary of the notation used in multiple linear regression:

| General Notation | Description | Python (if applicable) |
|:----------------:|:-----------:|:----------------------:|
|      $a$         | scalar, non-bold | |
|  $\mathbf{a}$   | vector, bold | |
|  $\mathbf{A}$   | matrix, bold capital | |
| **Regression** | | |
|  $\mathbf{X}$   | training example matrix | `X_train` |
|  $\mathbf{y}$   | training example targets | `y_train` |
| $\mathbf{x}^{(i)}$, $y^{(i)}$ | $i_{th}$ Training Example | `X[i]`, `y[i]` |
|        $m$       | number of training examples | `m` |
|        $n$       | number of features in each example | `n` |
|  $\mathbf{w}$   | parameter: weight vector | `w` |
|        $b$       | parameter: bias | `b` |
| $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ | Model prediction: $\mathbf{w} \cdot \mathbf{x}^{(i)}+b$ | `f_wb` |

The model's prediction with multiple variables is given by the linear model:

$$ f_{\mathbf{w},b}(\mathbf{x}) =  w_0x_0 + w_1x_1 +... + w_{n-1}x_{n-1} + b \tag{1}$$

or in **vector notation**:

$$ f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w} \cdot \mathbf{x} + b  \tag{2} $$ 

where $\cdot$ is a vector `dot product`.

The key difference from simple linear regression:
- $\mathbf{w}$ is now a **vector** with $n$ elements (one weight per feature), not a scalar.
- $\mathbf{x}^{(i)}$ is also a **vector** with $n$ elements.

<a name="4.3"></a>
### 4.3 Single Prediction (vectorized using np.dot)

Equation (1) can be implemented much more efficiently using the **dot product** as in equation (2).

NumPy `np.dot()` performs a vector dot product, which is both faster and more concise:

In [ ]:
def predict(x, w, b): 
    """
    Single prediction using linear regression (vectorized)
    Args:
      x (ndarray): Shape (n,) example with multiple features
      w (ndarray): Shape (n,) model parameters   
      b (scalar):             model parameter 
      
    Returns:
      p (scalar):  prediction
    """
    p = np.dot(x, w) + b     
    return p    

In [ ]:
# Verify: both methods should give the same result
x_vec = X_train[0, :]
f_wb_loop = predict_single_loop(x_vec, w_init, b_init)
f_wb_vec = predict(x_vec, w_init, b_init)

print(f"Loop prediction: {f_wb_loop:.4f}")
print(f"Vectorized prediction: {f_wb_vec:.4f}")
print(f"Both methods match: {np.isclose(f_wb_loop, f_wb_vec)}")

The results are the same! Going forward, we will use `np.dot` for efficiency. The prediction is now a **single statement** instead of a loop.

<a name="5"></a>
## 5 - Compute Cost

Gradient descent involves repeated steps to adjust the value of your parameters $(\mathbf{w}, b)$ to gradually get a smaller and smaller cost $J(\mathbf{w},b)$.

The cost function for multiple linear regression $J(\mathbf{w},b)$ is defined as:

$$J(\mathbf{w},b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})^2 \tag{3}$$

where:

$$ f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)} + b  \tag{4} $$ 

Note that $\mathbf{w}$ and $\mathbf{x}^{(i)}$ are **vectors** rather than scalars, supporting multiple features.

In [ ]:
def compute_cost(X, y, w, b): 
    """
    Computes the cost function for multiple linear regression.

    Args:
        X (ndarray (m,n)): Data, m examples with n features
        y (ndarray (m,)) : target values
        w (ndarray (n,)) : model parameters  
        b (scalar)       : model parameter

    Returns:
        total_cost (float): The cost of using w,b as the parameters for
                            linear regression to fit the data
    """
    m = X.shape[0]
    cost = 0.0
    for i in range(m):                                
        f_wb_i = np.dot(X[i], w) + b
        cost = cost + (f_wb_i - y[i])**2
    total_cost = cost / (2 * m)
    return total_cost

In [ ]:
# Compute cost with initial w and b = 0
initial_w = np.zeros(X_train.shape[1])
initial_b = 0

cost = compute_cost(X_train, y_train, initial_w, initial_b)
print(f'Cost at initial w (zeros), b (0): {cost:.3f}')

# Compute cost with near-optimal parameters
cost_opt = compute_cost(X_train, y_train, w_init, b_init)
print(f'Cost at near-optimal w, b: {cost_opt:.6e}')

<a name="6"></a>
## 6 - Gradient Descent

The gradient descent algorithm for **multiple variables** is:

$$\begin{align*} \text{repeat}& \text{ until convergence:} \; \lbrace \newline\;
& w_j = w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{5}  \; & \text{for j = 0..n-1}\newline
&b\ \ = b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}  \newline \rbrace
\end{align*}$$

where, $n$ is the number of features, parameters $w_j$,  $b$, are **updated simultaneously** and where  

$$
\begin{align}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \tag{6}  \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{7}
\end{align}
$$

* $m$ is the number of training examples in the data set
* $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ is the model's prediction, while $y^{(i)}$ is the target value

<a name="6.1"></a>
### 6.1 Compute Gradient

The implementation below calculates equations (6) and (7) using:
- An outer loop over all $m$ examples
  - $\frac{\partial J}{\partial b}$ is accumulated directly
  - An inner loop over all $n$ features computes $\frac{\partial J}{\partial w_j}$

In [ ]:
def compute_gradient(X, y, w, b): 
    """
    Computes the gradient for linear regression 
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters  
      b (scalar)       : model parameter
      
    Returns:
      dj_dw (ndarray (n,)): The gradient of the cost w.r.t. the parameters w. 
      dj_db (scalar):       The gradient of the cost w.r.t. the parameter b. 
    """
    m, n = X.shape           # (number of examples, number of features)
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range(m):                             
        err = (np.dot(X[i], w) + b) - y[i]   
        for j in range(n):                         
            dj_dw[j] = dj_dw[j] + err * X[i, j]    
        dj_db = dj_db + err                        
    dj_dw = dj_dw / m                                
    dj_db = dj_db / m                                
        
    return dj_dw, dj_db

In [ ]:
# Compute and display gradient with w initialized to zeroes
initial_w = np.zeros(X_train.shape[1])
initial_b = 0

tmp_dj_dw, tmp_dj_db = compute_gradient(X_train, y_train, initial_w, initial_b)
print('Gradient at initial w (zeros), b (0):')
print(f'  dj_dw: {tmp_dj_dw}')
print(f'  dj_db: {tmp_dj_db:.4f}')

<a name="6.2"></a>
### 6.2 Gradient Descent Implementation

Now let's put it all together. The `gradient_descent` function implements equation (5) above.

In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters): 
    """
    Performs batch gradient descent to learn w and b. Updates w and b by taking 
    num_iters gradient steps with learning rate alpha
    
    Args:
      X (ndarray (m,n))   : Data, m examples with n features
      y (ndarray (m,))    : target values
      w_in (ndarray (n,)) : initial model parameters  
      b_in (scalar)       : initial model parameter
      cost_function       : function to compute cost
      gradient_function   : function to compute the gradient
      alpha (float)       : Learning rate
      num_iters (int)     : number of iterations to run gradient descent
      
    Returns:
      w (ndarray (n,)) : Updated values of parameters 
      b (scalar)       : Updated value of parameter 
    """
    
    # An array to store cost J at each iteration primarily for graphing later
    J_history = []
    w = copy.deepcopy(w_in)  # avoid modifying global w within function
    b = b_in
    
    for i in range(num_iters):

        # Calculate the gradient and update the parameters
        dj_dw, dj_db = gradient_function(X, y, w, b)

        # Update Parameters using w, b, alpha and gradient
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
      
        # Save cost J at each iteration
        if i < 100000:      # prevent resource exhaustion 
            J_history.append(cost_function(X, y, w, b))

        # Print cost every at intervals 10 times or as many iterations if < 10
        if i % math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]:8.2f}   ")
        
    return w, b, J_history  # return final w,b and J history for graphing

<a name="7"></a>
## 7 - Gradient Descent on Raw Features

Let's first run gradient descent **WITHOUT** feature scaling to see the problem.

**Note**: With raw features, the learning rate must be kept very small (`α = 5e-7`). This is because the features have very different scales:
- Size ranges from ~600-3500 sqft
- Bedrooms: 1-5
- Floors: 1-2  
- Age: 1-100 years

The large feature magnitudes cause the `size` gradient to dominate, while `bedrooms` and `floors` barely move. A larger learning rate would cause the cost to **diverge** (increase instead of decrease).

In [ ]:
# Run gradient descent on RAW (unscaled) features
initial_w = np.zeros(X_train.shape[1])
initial_b = 0.

iterations_raw = 1000
alpha_raw = 5.0e-7  # Must be tiny to avoid divergence!

w_raw, b_raw, J_hist_raw = gradient_descent(X_train, y_train, initial_w, initial_b,
                                            compute_cost, compute_gradient, 
                                            alpha_raw, iterations_raw)
print(f"\nb,w found by gradient descent (raw): b = {b_raw:0.4f}, w = {w_raw}")
print(f"Final cost (raw features): {J_hist_raw[-1]:.2f}")

In [ ]:
# Plot cost vs iteration (raw features)
fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout=True, figsize=(12, 4))
ax1.plot(J_hist_raw)
ax2.plot(100 + np.arange(len(J_hist_raw[100:])), J_hist_raw[100:])
ax1.set_title("Cost vs. iteration (Raw Features)");  ax2.set_title("Cost vs. iteration (tail)")
ax1.set_ylabel('Cost')             ;  ax2.set_ylabel('Cost') 
ax1.set_xlabel('iteration step')   ;  ax2.set_xlabel('iteration step') 
plt.show()

print(f"\nCost is still {J_hist_raw[-1]:.2f} after {iterations_raw} iterations — not converged!")
print("The model is NOT accurate yet because features have very different scales.")

As you can see, the cost is **still very high** after 1000 iterations with the tiny learning rate. The model is far from converged.

This is because features have **very different scales** (size ~1000s vs floors ~1-2), causing uneven gradient updates. The solution is **Feature Scaling**!

<a name="8"></a>
## 8 - Feature Scaling

The importance of rescaling the dataset so the features have a similar range.

When features have very different ranges, gradient descent updates them unevenly:
- $\alpha$ is **shared** by all parameter updates ($w$'s and $b$)
- The error term is multiplied by each feature value for the $w$ updates
- Features with large values (like size ~1000s) get huge gradient updates
- Features with small values (like floors ~1-2) get tiny gradient updates

This makes it impossible to choose a good learning rate — too large and `size` diverges, too small and `bedrooms`/`floors` converge extremely slowly.

The solution is to **normalize** all features to have similar ranges.

<a name="8.1"></a>
### 8.1 Z-Score Normalization

After z-score normalization, all features will have a **mean of 0** and a **standard deviation of 1**.

To implement z-score normalization, adjust your input values as shown in this formula:

$$x^{(i)}_j = \dfrac{x^{(i)}_j - \mu_j}{\sigma_j} \tag{8}$$ 

where $j$ selects a feature or a column in the $\mathbf{X}$ matrix. $\mu_j$ is the mean of all the values for feature (j) and $\sigma_j$ is the standard deviation of feature (j).

$$
\begin{align}
\mu_j &= \frac{1}{m} \sum_{i=0}^{m-1} x^{(i)}_j \tag{9}\\
\sigma^2_j &= \frac{1}{m} \sum_{i=0}^{m-1} (x^{(i)}_j - \mu_j)^2  \tag{10}
\end{align}
$$

> **Implementation Note:** When normalizing the features, it is important to store the values used for normalization — the mean value and the standard deviation used for the computations. After learning the parameters from the model, we often want to predict the prices of houses we have not seen before. Given a new x value, we must **first normalize x using the same mean and standard deviation** that we had previously computed from the training set.

In [ ]:
def zscore_normalize_features(X):
    """
    Computes X, z-score normalized by column
    
    Args:
      X (ndarray (m,n))     : input data, m examples, n features
      
    Returns:
      X_norm (ndarray (m,n)): input normalized by column
      mu (ndarray (n,))     : mean of each feature
      sigma (ndarray (n,))  : standard deviation of each feature
    """
    # find the mean of each column/feature
    mu     = np.mean(X, axis=0)                 # mu will have shape (n,)
    # find the standard deviation of each column/feature
    sigma  = np.std(X, axis=0)                  # sigma will have shape (n,)
    # element-wise, subtract mu for that column from each example, divide by std for that column
    X_norm = (X - mu) / sigma      

    return (X_norm, mu, sigma)

<a name="8.2"></a>
### 8.2 Normalize the Training Data

Let's apply z-score normalization to our training data and see the effect:

In [ ]:
# Normalize the training features
X_norm, X_mu, X_sigma = zscore_normalize_features(X_train)

print("--- BEFORE normalization ---")
print(f"  X_train mean (per feature):  {np.mean(X_train, axis=0)}")
print(f"  X_train std  (per feature):  {np.std(X_train, axis=0)}")
print(f"  X_train range (per feature): {np.ptp(X_train, axis=0)}")

print("\n--- AFTER normalization ---")
print(f"  X_norm mean (per feature):   {np.mean(X_norm, axis=0)}")
print(f"  X_norm std  (per feature):   {np.std(X_norm, axis=0)}")
print(f"  X_norm range (per feature):  {np.ptp(X_norm, axis=0)}")

print(f"\nStored mu    = {X_mu}")
print(f"Stored sigma = {X_sigma}")
print("\nAll features now have mean ≈ 0 and std ≈ 1!")

In [ ]:
# Visualize the normalized features
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
for i in range(4):
    axes[i].scatter(X_norm[:, i], y_train, marker='x', c='b')
    axes[i].set_xlabel(feature_names[i] + ' (normalized)')
    if i == 0:
        axes[i].set_ylabel('Price (in $1000s)')
    axes[i].set_title(f'Price vs {feature_names[i]} (norm)')
plt.tight_layout()
plt.show()

print("After normalization, all features have a similar range — gradient descent will work much better!")

<a name="8.3"></a>
### 8.3 Gradient Descent with Normalized Features

Now let's run gradient descent again, but this time on the **normalized** features.

Because all features have the same scale, we can use a **much larger learning rate** (`α = 1.0e-1` vs `5.0e-7` before — that's **200,000× larger!**) and converge much faster:

In [ ]:
# Run gradient descent on NORMALIZED features
initial_w = np.zeros(X_norm.shape[1])
initial_b = 0.

iterations_norm = 1000
alpha_norm = 1.0e-1  # 200,000x larger than before!

w_norm, b_norm, J_hist_norm = gradient_descent(X_norm, y_train, initial_w, initial_b,
                                               compute_cost, compute_gradient, 
                                               alpha_norm, iterations_norm)
print(f"\nb,w found by gradient descent (normalized): b = {b_norm:0.4f}, w = {w_norm}")
print(f"Final cost (normalized features): {J_hist_norm[-1]:.4f}")
print(f"\nCompare: Raw features cost = {J_hist_raw[-1]:.2f}")
print(f"         Normalized cost    = {J_hist_norm[-1]:.4f}")
print(f"         Improvement ratio  = {J_hist_raw[-1] / J_hist_norm[-1]:.0f}x lower cost!")

<a name="8.4"></a>
### 8.4 Convergence Comparison

Let's compare the convergence plots — raw features vs. normalized features:

In [ ]:
fig, axes = plt.subplots(1, 3, constrained_layout=True, figsize=(18, 4))

# Plot 1: Raw feature cost
axes[0].plot(J_hist_raw, 'r-', label='Raw features')
axes[0].set_title(f'Raw Features (α={alpha_raw})')
axes[0].set_ylabel('Cost')
axes[0].set_xlabel('Iteration')
axes[0].legend()

# Plot 2: Normalized feature cost
axes[1].plot(J_hist_norm, 'b-', label='Normalized features')
axes[1].set_title(f'Normalized Features (α={alpha_norm})')
axes[1].set_ylabel('Cost')
axes[1].set_xlabel('Iteration')
axes[1].legend()

# Plot 3: Normalized tail (zoomed in)
axes[2].plot(100 + np.arange(len(J_hist_norm[100:])), J_hist_norm[100:], 'b-')
axes[2].set_title('Normalized (tail — zoomed in)')
axes[2].set_ylabel('Cost')
axes[2].set_xlabel('Iteration')

plt.show()

print(f"With feature scaling, cost dropped from {J_hist_norm[0]:.2f} to {J_hist_norm[-1]:.4f}")
print(f"Without feature scaling, cost only dropped from {J_hist_raw[0]:.2f} to {J_hist_raw[-1]:.2f}")
print("\n→ Feature scaling enables DRAMATICALLY faster and better convergence!")

In [ ]:
# Print optimized predictions vs actuals for the first 10 examples
print("Predictions vs Actual values (first 10 examples, NORMALIZED model):")
print(f"{'Example':>8}  {'Prediction':>12}  {'Actual':>10}  {'Error':>10}")
print("-" * 45)
for i in range(min(10, X_norm.shape[0])):
    pred = np.dot(X_norm[i], w_norm) + b_norm
    err = pred - y_train[i]
    print(f"{i:>8}  {pred:>12.2f}  {y_train[i]:>10.2f}  {err:>10.2f}")

**The predictions are now much more accurate!** The normalized model closely matches the actual house prices.

<a name="9"></a>
## 9 - Predictions

Now we can use our optimized model to make predictions on **new houses**.

**Important:** Since the model was trained on normalized features, we must also normalize the input features using the **same `mu` and `sigma`** from the training set before making predictions.

In [ ]:
# Prediction 1: A 1200 sqft house, 3 bedrooms, 1 floor, 40 years old
x_test1 = np.array([1200, 3, 1, 40])
x_test1_norm = (x_test1 - X_mu) / X_sigma  # Normalize using training stats!
predict1 = np.dot(x_test1_norm, w_norm) + b_norm
print('For a 1200 sqft, 3 bed, 1 floor, 40 yr old house, we predict a price of $%.2f' % (predict1 * 1000))

# Prediction 2: A 2000 sqft house, 4 bedrooms, 2 floors, 10 years old
x_test2 = np.array([2000, 4, 2, 10])
x_test2_norm = (x_test2 - X_mu) / X_sigma
predict2 = np.dot(x_test2_norm, w_norm) + b_norm
print('For a 2000 sqft, 4 bed, 2 floor, 10 yr old house, we predict a price of $%.2f' % (predict2 * 1000))

# Prediction 3: A 800 sqft house, 2 bedrooms, 1 floor, 70 years old
x_test3 = np.array([800, 2, 1, 70])
x_test3_norm = (x_test3 - X_mu) / X_sigma
predict3 = np.dot(x_test3_norm, w_norm) + b_norm
print('For a 800 sqft, 2 bed, 1 floor, 70 yr old house, we predict a price of $%.2f' % (predict3 * 1000))

# Prediction 4: A 3000 sqft house, 4 bedrooms, 2 floors, 5 years old
x_test4 = np.array([3000, 4, 2, 5])
x_test4_norm = (x_test4 - X_mu) / X_sigma
predict4 = np.dot(x_test4_norm, w_norm) + b_norm
print('For a 3000 sqft, 4 bed, 2 floor, 5 yr old house, we predict a price of $%.2f' % (predict4 * 1000))

## Congratulations!

In this experiment you:
- Extended the data structures and routines from simple linear regression to support **multiple features**
- Loaded a **real dataset** with 200 training examples and 4 features
- Implemented prediction both **element by element** (loop) and **vectorized** (using `np.dot`)
- Implemented the **cost function** for multiple linear regression
- Implemented **gradient computation** with multiple variables 
- Implemented **gradient descent** to learn optimal parameters
- Demonstrated the problem with **unscaled features** — slow convergence and inaccurate predictions
- Implemented **z-score normalization** to rescale features to similar ranges
- Showed that feature scaling enables a **200,000× larger learning rate** and dramatically better convergence
- Used the optimized model to make **accurate predictions** on new houses
- Utilized NumPy `np.dot` to vectorize implementations for **speed and simplicity**